# 🌾 Paddy-YOLO: Proposed Architecture Training on R2000

This notebook trains the actual **Paddy-YOLO** model (RGG Backbone + Dynamic Inner-MPDIoU / DIMIoU Loss) on the **R2000** rice pest dataset (`dmrashidferrer/r2000v1-2024-images`).

### Architectural Innovations (Zheng et al., 2025):
- **Backbone**: 4 RGG blocks replacing C2f in YOLOv8-N (RepConv + dual GhostConv with 5x5 DWConv + feature concatenation).
- **Bounding Box Loss**: Dynamic Inner-MPDIoU (**DIMIoU**, Strategy H cosine scheduling).
- **Optimizer**: AdamW, lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=0.0005, 300 epochs.
- **Evaluation**: 5-Fold Cross Validation Fold 0 (3 train / 1 val / 1 test).

In [ ]:
# Cell 1: Extract project codebase
import base64, io, zipfile, os

B64_ARCHIVE = '''...'''

zip_data = base64.b64decode(B64_ARCHIVE.encode('utf-8'))
with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
    z.extractall('.')

print('Codebase successfully unpacked into working directory.')


In [ ]:
# Cell 2: Verify GPU Accelerator & PyTorch Environment
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('Device Count:', torch.cuda.device_count())
else:
    raise SystemError('GPU accelerator is not available! Please enable GPU in notebook settings.')


In [ ]:
# Cell 3: Install Ultralytics and Run Sanity Check
!pip install ultralytics -q
!python verify_paddy_yolo.py --smoke-test


In [ ]:
# Cell 4: Discover & Prepare Kaggle R2000 Dataset with 5-Fold Splits
import os, subprocess
from pathlib import Path
from paddy_yolo.dataset.prepare_r2000 import prepare_kaggle_r2000, create_5fold_splits

kaggle_input = Path('/kaggle/input')
data_yamls = list(kaggle_input.rglob('data.yaml')) if kaggle_input.exists() else []

if data_yamls:
    raw_data_dir = data_yamls[0].parent
    print(f'Discovered R2000 dataset via data.yaml at: {raw_data_dir}')
else:
    imgs = list(kaggle_input.rglob('*.jpg')) if kaggle_input.exists() else []
    if imgs:
        raw_data_dir = imgs[0].parent.parent
        print(f'Discovered R2000 dataset via image search at: {raw_data_dir}')
    else:
        print('Dataset not mounted in /kaggle/input. Downloading via Kaggle API...')
        download_dir = Path('/kaggle/working/data/raw_r2000')
        download_dir.mkdir(parents=True, exist_ok=True)
        subprocess.run([
            'kaggle', 'datasets', 'download',
            '-d', 'dmrashidferrer/r2000v1-2024-images',
            '--unzip', '-p', str(download_dir)
        ], check=True)
        raw_data_dir = download_dir
        print(f'Downloaded and unzipped dataset to: {raw_data_dir}')

output_dir = Path('/kaggle/working/data/r2000')
prepare_kaggle_r2000(input_dir=raw_data_dir, output_dir=output_dir)
folds = create_5fold_splits(dataset_dir=output_dir)
print('Generated 5-fold cross validation split configs:')
for f in folds:
    print(f'  -> {f}')


In [ ]:
# Cell 5: Train Actual Paddy-YOLO (RGG Backbone + DIMIoU Strategy H, 300 Epochs)
!python train.py \
    --model configs/paddy-yolo.yaml \
    --data /kaggle/working/data/r2000/data_fold0.yaml \
    --hyp configs/hyp_paper.yaml \
    --epochs 300 \
    --batch 32 \
    --imgsz 640 \
    --strategy H \
    --device 0 \
    --project /kaggle/working/runs/train \
    --name paddy_yolo_fold0


In [ ]:
# Cell 6: Evaluate Paddy-YOLO Checkpoint on Test Set
best_weights = '/kaggle/working/runs/train/paddy_yolo_fold0/weights/best.pt'

!python val.py \
    --weights {best_weights} \
    --data /kaggle/working/data/r2000/data_fold0.yaml \
    --split test \
    --device 0


In [ ]:
# Cell 7: Package artifacts for download
import shutil
train_run_dir = '/kaggle/working/runs/train/paddy_yolo_fold0'
zip_dest = '/kaggle/working/paddy_yolo_fold0_results'
if os.path.exists(train_run_dir):
    shutil.make_archive(zip_dest, 'zip', train_run_dir)
    print(f'Created downloadable results archive at: {zip_dest}.zip')
